## 🛠️ Step 1: Environment Setup
We begin by installing the required library packages for the dashboard application.

In [4]:
#@title Install Requirements
!pip install -q streamlit plotly scipy scikit-learn

In [3]:
#@title Retrieve Workspace Assets from Drive
import os
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Configure source paths
gdrive_source = '/content/drive/MyDrive/Global_Trust_Bank_Dashboard'
target_dir = '/content'

# Check alternative folder if primary is missing
if not os.path.exists(gdrive_source):
    gdrive_source = '/content/drive/MyDrive/Global trust bank'

if os.path.exists(gdrive_source):
    print(f"📦 Source found at: {gdrive_source}")
    files_to_copy = os.listdir(gdrive_source)
    print(f"📄 Transferring files: {files_to_copy}")
    for file_name in files_to_copy:
        src_path = os.path.join(gdrive_source, file_name)
        dst_path = os.path.join(target_dir, file_name)
        if os.path.isdir(src_path):
            if os.path.exists(dst_path):
                shutil.rmtree(dst_path)
            shutil.copytree(src_path, dst_path)
        else:
            shutil.copy2(src_path, dst_path)
    print("✅ All dashboard assets transferred successfully!")
else:
    print(f"❌ Could not find source directory at either backup path in your Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Source found at: /content/drive/MyDrive/Global_Trust_Bank_Dashboard
📄 Transferring files: ['app.py', 'requirements.txt', 'high_risk_customers.csv', 'Customer-Churn-Records.csv']
✅ All dashboard assets transferred successfully!


## 🚀 Step 3: Run the Streamlit Server
Now we start the backend Streamlit application using `app.py` in headless mode.

In [7]:
#@title Launch Streamlit in Background
import os, subprocess, time

os.chdir("/content")
assert os.path.exists("app.py"), "app.py is missing! Run Step 2 (Google Drive transfer) again."

# Terminate any old instances
subprocess.run("pkill -f 'streamlit run' || true", shell=True)

# Run Streamlit
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true "
                 "--server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1", shell=True)
time.sleep(5)

print("🟢 Streamlit server is running. Log output preview:")
print("-----------------------------------------------")
print(open("streamlit.log").read()[-500:])

🟢 Streamlit server is running. Log output preview:
-----------------------------------------------


2026-11-15 08:26:21.869 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.154.42.208:8501




## 🔗 Step 4: Expose Server with Cloudflare Tunnel
Create a secure, private tunnel to connect to your dashboard directly using `trycloudflare`.

In [6]:
#@title Generate Public trycloudflare Tunnel Link
import re, os, subprocess, time, urllib.request

if not os.path.exists("cloudflared"):
    print("📥 Downloading Cloudflare binary...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)

# Terminate and clear old tunnels completely
subprocess.run("pkill -f 'cloudflared' || true", shell=True)
time.sleep(2)

if os.path.exists("tunnel.log"):
    os.remove("tunnel.log")

# Open a fresh public tunnel on Streamlit's port 8501
subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 --no-autoupdate > tunnel.log 2>&1", shell=True)

print("🔄 Initializing public tunnel secure link...")
new_link = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        log_content = open("tunnel.log").read()
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", log_content)
        if m:
            new_link = m.group(0)
            break

if new_link:
    print("\n" + "="*60)
    print(f"🎉 Click here to view your live dashboard: {new_link}")
    print("="*60)
else:
    print("❌ Tunnel generation timed out. See logs:")
    if os.path.exists("tunnel.log"):
        print(open("tunnel.log").read()[-500:])
    else:
        print("Log file not generated.")

📥 Downloading Cloudflare binary...
🔄 Initializing public tunnel secure link...

🎉 Click here to view your live dashboard: https://calls-orbit-including-guest.trycloudflare.com
